In [1]:
import pandas as pd 
import numpy as np
import matplotlib as plt
import seaborn as sns
from sqlalchemy import create_engine, text


In [2]:
df = pd.read_csv("sales.csv", encoding='utf-8', encoding_errors='replace')

In [3]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
# load_superstore_to_mysql.py
# Usage:
# 1) create a .env file with DB creds (example below),
# 2) put superstore.csv in same folder,
# 3) run: python load_superstore_to_mysql.py

import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text

# --------- CONFIG (use .env or edit here) -------------
# Create a .env file in the same folder with:
# DB_USER=root
# DB_PASS=your_password
# DB_HOST=127.0.0.1
# DB_PORT=3306
# DB_NAME=superstore_db
# CSV_PATH=superstore.csv
load_dotenv()

DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS", "your_password")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = int(os.getenv("DB_PORT", 3306))
DB_NAME = os.getenv("DB_NAME", "superstore_db")
CSV_PATH = os.getenv("CSV_PATH", "superstore.csv")
# ------------------------------------------------------

def get_engine():
    # Use pymysql driver
    engine_str = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
    return create_engine(engine_str, pool_pre_ping=True)

def main():
    print("1) Loading CSV into pandas...")
    # parse dates - these column names must match your CSV header exactly
    df = pd.read_csv(sales.csv, parse_dates=["Order Date", "Ship Date"], dayfirst=False)

    print(f"   Rows loaded: {len(df)}")
    # standard cleaning
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]  # drop unnamed cols
    # rename Row ID -> RowID to be friendly with SQL column name
    if "Row ID" in df.columns:
        df = df.rename(columns={"Row ID": "RowID"})
    # ensure RowID exists and is integer
    if "RowID" not in df.columns:
        df.insert(0, "RowID", range(1, len(df) + 1))
    df["RowID"] = df["RowID"].astype(int)

    # Coerce numeric columns
    for col in ["Sales", "Profit", "Quantity", "Discount"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Optional: drop Customer Name to avoid PII if you don't need it
    # Uncomment next line to drop it
    # if "Customer Name" in df.columns: df = df.drop(columns=["Customer Name"])

    # Connect to MySQL
    engine = get_engine()
    print("2) Connecting to MySQL and uploading table (this will REPLACE existing 'orders' table)...")
    # to_sql with if_exists='replace' will create table — but with generic types.
    # If you ran create_schema.sql earlier and want to keep types, set if_exists='append' instead.
    df.to_sql(name="orders", con=engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    print("   Upload complete.")

    # Create indexes if table was replaced (safe to run; may error on some MySQL versions but not fatal)
    print("3) Ensuring basic indexes exist (may show errors if indexes already exist)...")
    idx_sql = [
        "CREATE INDEX idx_order_date ON orders(`Order Date`);",
        "CREATE INDEX idx_order_id ON orders(`Order ID`);",
        "CREATE INDEX idx_customer ON orders(`Customer ID`);",
        "CREATE INDEX idx_product ON orders(`Product ID`);",
        "CREATE INDEX idx_state ON orders(State);"
    ]
    with engine.connect() as conn:
        for q in idx_sql:
            try:
                conn.execute(text(q))
            except Exception as e:
                # ignore index creation errors (index may already exist or syntax)
                print("   (index create issue ignored):", e)

    # Quick sanity check query to confirm
    print("4) Running sanity-check query: total rows and sample aggregates...")
    with engine.connect() as conn:
        total = conn.execute(text("SELECT COUNT(*) AS cnt FROM orders;")).fetchone()[0]
        print(f"   Rows in DB table 'orders': {total}")
        sample = conn.execute(text("SELECT YEAR(`Order Date`) AS yr, SUM(Sales) AS total_sales FROM orders GROUP BY yr ORDER BY yr;")).fetchall()
        print("   Sales by year (sample):")
        for row in sample:
            print("    ", row[0], round(row[1],2))

    print("DONE. You can now run SQL queries on database 'superstore_db', table 'orders'.")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'dotenv'